# Hunyuan3D-2 — Zombie Game Asset Generator

Generates concept images, then turns them into 3D models (`.glb`) using **Hunyuan3D-2** (Tencent's image-to-3D model). Output `.glb` files are meant to be dropped into `godot/assets/generated_models/` for the zombie survival prototype.

**Before running:** in the Kaggle notebook settings (right sidebar) set **Accelerator = GPU T4 x2** (or P100). No Kaggle API token is needed to *run* this notebook — a token is only needed if you want to `kaggle kernels push` it from the command line (see `kaggle/README.md`).

In [ ]:
import torch, subprocess, sys, os
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise SystemExit('No GPU detected — enable GPU T4 x2 in Notebook Settings > Accelerator, then re-run.')

## Part 1 — Generate concept images

Quick text-to-image pass (SD-Turbo, fast + light on VRAM) to get clean reference images for the zombie, a building, and a street prop. Skip this cell and upload your own images to `/kaggle/working/concept_images/` instead if you'd rather use hand-picked references.

In [ ]:
!pip install -q diffusers accelerate safetensors

In [ ]:
os.makedirs('/kaggle/working/concept_images', exist_ok=True)

prompts = {
    'zombie_walker': 'full body concept art of a shambling zombie walker character, T-pose, plain grey background, centered, game asset',
    'zombie_crawler': 'full body concept art of a crawling mutilated zombie creature, T-pose limbs spread, plain grey background, centered, game asset',
    'zombie_bloated': 'full body concept art of a bloated toxic zombie character, T-pose, plain grey background, centered, game asset',
    'zombie_screamer': 'full body concept art of a screaming feral zombie character, T-pose, plain grey background, centered, game asset',
    'skeleton_warrior': 'full body concept art of an armored skeleton warrior character, T-pose, plain grey background, centered, game asset',
    'ghoul_creature': 'full body concept art of a hunched ghoul monster creature, T-pose, plain grey background, centered, game asset',
    'grave_headstone': 'concept art of a cracked stone grave headstone, single object, plain white background, centered, game asset',
    'cemetery_gate': 'concept art of a rusty wrought iron cemetery gate, single object, plain white background, centered, game asset',
    'hanging_cage': 'concept art of a rusty hanging iron cage, single object, plain white background, centered, game asset',
    'blood_altar': 'concept art of a stone ritual blood altar, single object, plain white background, centered, game asset',
    'torture_rack': 'concept art of a wooden torture rack, single object, plain white background, centered, game asset',
    'broken_coffin': 'concept art of a broken wooden coffin, single object, plain white background, centered, game asset',
    'haunted_lantern': 'concept art of an old rusty haunted oil lantern, single object, plain white background, centered, game asset',
    'ritual_candle_circle': 'concept art of a ring of melted ritual candles on the ground, single object, plain white background, centered, game asset',
    'crow_perched': 'concept art of a perched black crow, single object, plain white background, centered, game asset',
    'creepy_scarecrow': 'full body concept art of a creepy straw scarecrow, T-pose, plain grey background, centered, game asset',
    'abandoned_wheelchair': 'concept art of an old rusty abandoned wheelchair, single object, plain white background, centered, game asset',
    'rusty_chainsaw': 'concept art of a rusty old chainsaw prop, single object, plain white background, centered, game asset',
    'butcher_hook': 'concept art of a hanging rusty butcher meat hook, single object, plain white background, centered, game asset',
    'possessed_doll': 'full body concept art of a creepy possessed porcelain doll, T-pose, plain grey background, centered, game asset',
    'wooden_crate': 'concept art of a wooden storage crate, single object, plain white background, centered, game asset',
    'metal_barrel': 'concept art of a rusty metal oil barrel, single object, plain white background, centered, game asset',
    'wooden_barricade': 'concept art of a wooden street barricade, single object, plain white background, centered, game asset',
    'street_lamp': 'concept art of an old street lamp post, single object, plain white background, centered, game asset',
    'park_bench': 'concept art of a wooden park bench, single object, plain white background, centered, game asset',
    'dead_tree': 'concept art of a bare dead tree, single object, plain white background, centered, game asset',
    'rock_boulder': 'concept art of a large rock boulder, single object, plain white background, centered, game asset',
    'dumpster': 'concept art of a metal street dumpster, single object, plain white background, centered, game asset',
    'traffic_cone': 'concept art of an orange traffic cone, single object, plain white background, centered, game asset',
    'chain_link_fence': 'concept art of a section of chain link fence, single object, plain white background, centered, game asset',
    'brick_house': 'concept art of a small abandoned brick house, single object, plain white background, centered, game asset',
    'wooden_shack': 'concept art of a small wooden shack, single object, plain white background, centered, game asset',
    'watch_tower': 'concept art of a wooden survivor watch tower, single object, plain white background, centered, game asset',
    'sandbag_wall': 'concept art of a stacked sandbag wall barrier, single object, plain white background, centered, game asset',
    'military_jeep': 'concept art of an old military jeep vehicle, single object, plain white background, centered, game asset',
    'supply_box': 'concept art of a military supply box crate, single object, plain white background, centered, game asset',
    'first_aid_kit': 'concept art of a first aid medical kit box, single object, plain white background, centered, game asset',
    'ammo_crate': 'concept art of a wooden ammo crate, single object, plain white background, centered, game asset',
    'wooden_pallet': 'concept art of a wooden shipping pallet, single object, plain white background, centered, game asset',
    'wrecked_car': 'concept art of an old wrecked rusty car, single object, plain white background, centered, game asset',
}

for name, prompt in prompts.items():
    image = img_pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
    path = f'/kaggle/working/concept_images/{name}.png'
    image.save(path)
    print('saved', path)

del img_pipe
torch.cuda.empty_cache()

## Part 2 — Set up Hunyuan3D-2

Clones the official repo and installs its requirements. The optional texture-painting extensions (`custom_rasterizer`, `differentiable_renderer`) need a CUDA build step that can fail on some Kaggle images — that's fine, the notebook falls back to untextured/vertex-colored mesh export if they don't build.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd Hunyuan3D-2
!pip install -q -r requirements.txt

In [ ]:
import subprocess

def try_build(path, cmd):
    try:
        subprocess.run(cmd, cwd=path, shell=True, check=True)
        print(f'built OK: {path}')
        return True
    except subprocess.CalledProcessError as e:
        print(f'skipping optional build ({path}): {e}')
        return False

texture_available = try_build(
    '/kaggle/working/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer',
    'python3 setup.py install'
)
if texture_available:
    texture_available = try_build(
        '/kaggle/working/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer',
        'bash compile_mesh_painter.sh'
    )

## Part 3 — Image → 3D mesh

Runs each concept image through the Hunyuan3D-2 shape pipeline and exports a `.glb` per asset.

In [ ]:
import sys, os, glob
sys.path.insert(0, '/kaggle/working/Hunyuan3D-2')

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')

os.makedirs('/kaggle/working/generated_models', exist_ok=True)

image_paths = sorted(glob.glob('/kaggle/working/concept_images/*.png'))
meshes = {}

for path in image_paths:
    name = os.path.splitext(os.path.basename(path))[0]
    print('generating mesh for', name)
    mesh = shape_pipeline(image=path)[0]
    out_path = f'/kaggle/working/generated_models/{name}.glb'
    mesh.export(out_path)
    meshes[name] = mesh
    print('saved', out_path)

In [ ]:
# Optional: PBR texture painting pass (only runs if the rasterizer built above).
if texture_available:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
    for path in image_paths:
        name = os.path.splitext(os.path.basename(path))[0]
        textured = paint_pipeline(meshes[name], image=path)
        out_path = f'/kaggle/working/generated_models/{name}_textured.glb'
        textured.export(out_path)
        print('saved', out_path)
else:
    print('Texture painting skipped (rasterizer did not build) — using the plain meshes from Part 3.')

## Part 4 — Package the output

Zips everything in `generated_models/` so it can be downloaded from the notebook's Output tab and dropped into `godot/assets/generated_models/` in the Godot project (see that folder's `README.md` for the import steps).

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/generated_models', 'zip', '/kaggle/working/generated_models')
print('Download: generated_models.zip from the Output tab.')